# ODI Data Preprocessing & Feature Engineering

**Google Colab-ready notebook**

This notebook completes all 7 requested steps:
1. EDA
2. Missing-value imputation
3. Categorical encoding
4. Numerical scaling
5. Feature engineering
6. Leakage-safe train/test split
7. CSV export + report/visualizations

> The `High_Average` target is created only to demonstrate a machine-learning pipeline.


In [ ]:
# Install/verify required libraries in Google Colab
!pip -q install pandas numpy matplotlib scikit-learn


In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# If the ZIP is extracted into /content, this path works:
PROJECT_ROOT = Path("/content/ODI_Preprocessing_GitHub_Project")

# Fallback for direct notebook upload/testing:
if not (PROJECT_ROOT / "data" / "raw_dataset.csv").exists():
    PROJECT_ROOT = Path("/content")

DATA_DIR = PROJECT_ROOT / "data"
VIZ_DIR = PROJECT_ROOT / "visualizations"
DOCS_DIR = PROJECT_ROOT / "docs"

DATA_DIR.mkdir(exist_ok=True)
VIZ_DIR.mkdir(exist_ok=True)
DOCS_DIR.mkdir(exist_ok=True)

raw_path = DATA_DIR / "raw_dataset.csv"

# If raw_dataset.csv is not found, upload it from your computer:
if not raw_path.exists():
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    raw_path = Path("/content") / uploaded_name

print("Using:", raw_path)


## Step 1 — Load the raw dataset and perform initial EDA

In [ ]:
df_raw = pd.read_csv(raw_path)

print("Shape:", df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes.to_frame("Data Type"))
display(df_raw.isna().sum().to_frame("Missing Values"))

print("Duplicate rows:", df_raw.duplicated().sum())


In [ ]:
# Replace common missing-value placeholders with NaN
df = df_raw.copy()
df = df.replace(["-", "—", "", " "], np.nan)

# Remove Excel-style empty/index columns
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed:")], errors="ignore")

print("After removing empty/index-like columns:", df.shape)
display(df.isna().sum().sort_values(ascending=False).to_frame("Missing Values"))


### Outlier identification with the IQR rule

Potential outliers are identified, not automatically deleted, because extreme cricket statistics may be genuine.


In [ ]:
# Convert numeric-looking columns
numeric_cols = ["Mat", "Inns", "NO", "Runs", "Ave", "BF", "SR", "100", "50", "0"]
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["HS"] = pd.to_numeric(df["HS"].astype(str).str.replace("*", "", regex=False), errors="coerce")

outlier_counts = {}
for c in df.select_dtypes(include=np.number).columns:
    s = df[c].dropna()
    if len(s) == 0:
        outlier_counts[c] = 0
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outlier_counts[c] = int(((s < low) | (s > high)).sum())

display(pd.Series(outlier_counts).sort_values(ascending=False).to_frame("IQR Outlier Count"))


In [ ]:
plt.figure(figsize=(9,5))
plt.hist(df["Runs"].dropna(), bins=30)
plt.title("Runs Distribution Before Scaling")
plt.xlabel("Runs")
plt.ylabel("Players")
plt.show()


In [ ]:
plt.figure(figsize=(9,5))
plt.boxplot(df["Runs"].dropna(), vert=False)
plt.title("Runs Boxplot Before Scaling")
plt.xlabel("Runs")
plt.show()


## Step 2 — Handle missing values

`-` has already been converted to NaN. The actual imputation is performed inside the Scikit-learn pipeline later:
- median for numeric variables
- most frequent value (mode) for categorical variables


## Step 3 — Encode categorical variables

The `Player` field is high-cardinality, so it is treated as an identifier. A `Team` feature is extracted from the text in `Player` and One-Hot Encoded as nominal categorical data.


In [ ]:
# Feature engineering from Span and Player
span_parts = df["Span"].astype(str).str.extract(r"(?P<Start_Year>\d{4})-(?P<End_Year>\d{4})")
df["Start_Year"] = pd.to_numeric(span_parts["Start_Year"], errors="coerce")
df["End_Year"] = pd.to_numeric(span_parts["End_Year"], errors="coerce")
df["Career_Years"] = df["End_Year"] - df["Start_Year"] + 1

df["Team"] = df["Player"].astype(str).str.extract(r"\((.*?)\)")[0].str.split("/").str[-1]
df["Team"] = df["Team"].replace("nan", np.nan)

display(df[["Player", "Span", "Team", "Start_Year", "End_Year", "Career_Years"]].head())


## Step 4 & 5 — Scale numeric features and engineer a predictive feature

In [ ]:
# New predictive feature
df["Runs_per_Innings"] = df["Runs"] / df["Inns"].replace(0, np.nan)

# Demonstration target based on the median batting average
ave_median = df["Ave"].median()
df["High_Average"] = (df["Ave"] >= ave_median).astype(int)

# Remove identifiers/raw target source from model features
model_df = df.drop(columns=["Player", "Span", "Ave"], errors="ignore")

display(model_df.head())
print("Target distribution:")
display(model_df["High_Average"].value_counts(normalize=True))


## Step 6 — Split first, then fit preprocessing only on training data

This is important for preventing **data leakage**. The scaler, imputer and encoder are fitted only on `X_train`.


In [ ]:
target = "High_Average"
X = model_df.drop(columns=[target])
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Train:", X_train.shape, "Test:", X_test.shape)


In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred, zero_division=0),
    "Recall": recall_score(y_test, pred, zero_division=0),
    "F1 Score": f1_score(y_test, pred, zero_division=0)
}
print(metrics)


### Visualize the same feature after StandardScaler

In [ ]:
scaler_demo = StandardScaler()
runs_train = X_train["Runs"].dropna().to_numpy().reshape(-1, 1)
scaled_runs = scaler_demo.fit_transform(runs_train).ravel()

plt.figure(figsize=(9,5))
plt.hist(scaled_runs, bins=30)
plt.title("Runs Distribution After StandardScaler")
plt.xlabel("Standardized Runs (z-score)")
plt.ylabel("Players")
plt.show()


## Step 7 — Export the final dataset and save project outputs

In [ ]:
processed_path = DATA_DIR / "processed_dataset.csv"
model_df.to_csv(processed_path, index=False)

# Save visualizations
plt.figure(figsize=(9,5))
plt.hist(df["Runs"].dropna(), bins=30)
plt.title("Runs Distribution Before Scaling")
plt.xlabel("Runs")
plt.ylabel("Players")
plt.tight_layout()
plt.savefig(VIZ_DIR / "01_runs_before_scaling.png", dpi=160)
plt.close()

plt.figure(figsize=(9,5))
plt.boxplot(df["Runs"].dropna(), vert=False)
plt.title("Runs Boxplot Before Scaling")
plt.xlabel("Runs")
plt.tight_layout()
plt.savefig(VIZ_DIR / "02_runs_boxplot_before_scaling.png", dpi=160)
plt.close()

plt.figure(figsize=(9,5))
plt.hist(scaled_runs, bins=30)
plt.title("Runs Distribution After StandardScaler")
plt.xlabel("Standardized Runs (z-score)")
plt.ylabel("Players")
plt.tight_layout()
plt.savefig(VIZ_DIR / "03_runs_after_scaling.png", dpi=160)
plt.close()

print("Saved:", processed_path)
print("Files in data:", list(DATA_DIR.iterdir()))
print("Files in visualizations:", list(VIZ_DIR.iterdir()))


## Model-performance interpretation

The Logistic Regression metrics above demonstrate that the final preprocessing pipeline produces model-ready features. The important methodological result is that imputation, encoding and scaling are learned from the training set only. This prevents test-set information from leaking into the model.

For the internship submission, include:
- this notebook
- `data/processed_dataset.csv`
- `README.md`
- `docs/preprocessing_report.md` or the PDF report
- all three visualization files
